# V3 — Feature Correctness Audit (Fixed)

This is the **single source of truth** for V3 Feature Correctness.

The notebook audits the final combined behavioral model population across **all observed vintages at once**. It does not process vintage-by-vintage.

### Scope

1. Feature inventory / schema consistency
2. Freddie Mac / SFLLD field-domain checks
3. Field-specific sentinel semantics
4. Missing vs documented unavailable/unknown states
5. Derived-feature identities
6. Categorical domain checks
7. Behavioral grain / duplicates
8. Point-in-time sanity checks
9. Cross-vintage data sanity
10. One consolidated PASS / REVIEW / FAIL decision

**Important:** this notebook only audits correctness. Correlation, redundancy, PCA, and feature reduction are deliberately excluded.


## 0. Audit philosophy

A feature is considered correct only when:

- its source meaning is understood;
- valid values and special codes follow the source documentation;
- missing / unavailable / unknown states are not conflated;
- derived values are internally consistent;
- predictors are available at the observation timestamp;
- the natural modelling grain is correct;
- and observed cross-vintage behavior does not indicate a silent semantic break.

A documentation uncertainty is **REVIEW**, not silently treated as PASS.


In [64]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import math
import warnings

import numpy as np
import pandas as pd

from pyspark.sql import DataFrame, SparkSession, functions as F

warnings.filterwarnings("ignore")

# Run from the repo root, or from a notebook located directly under the repo.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    if (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError(
            "Could not locate repository root. Set PROJECT_ROOT manually."
        )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)


Project root: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [65]:
from credit_risk.utils.config import read_config

config = read_config(PROJECT_ROOT)

parameters = config["parameters"]
modelling_config = parameters["modelling"]
features_config = modelling_config["features"]

APPROACH = parameters["modelling_approach"]
TARGET = parameters["target"]["name"]

NUMERIC_FEATURES = features_config.get("numerical_features", [])
CATEGORICAL_FEATURES = features_config.get("categorical_features", [])
ENGINEERED_FEATURES = features_config.get("engineered_features", [])
MODEL_FEATURES = list(dict.fromkeys(
    NUMERIC_FEATURES + CATEGORICAL_FEATURES + ENGINEERED_FEATURES
))

TRAIN_VINTAGES = modelling_config.get("vintages_train", [])
VALIDATION_VINTAGES = modelling_config.get("vintages_test", [])
OOT_VINTAGES = modelling_config.get("vintages_oot", [])
CONFIGURED_ALL_VINTAGES = parameters["data"].get("all_vintages", [])

print("Approach:", APPROACH)
print("Target:", TARGET)
print("Configured all vintages:", CONFIGURED_ALL_VINTAGES)
print("Train:", TRAIN_VINTAGES)
print("Validation:", VALIDATION_VINTAGES)
print("OOT:", OOT_VINTAGES)
print("Model features:", len(MODEL_FEATURES))


Approach: behavioral
Target: future_90dpd_12m
Configured all vintages: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Train: [2015, 2016, 2017, 2018]
Validation: [2019, 2020]
OOT: [2021, 2022]
Model features: 34


In [66]:
spark = (
    SparkSession.builder
    .appName("v3_feature_correctness_audit_fixed")
    .master(parameters["spark"]["master"])
    .config("spark.driver.memory", parameters["spark"]["driver_memory"])
    .config("spark.sql.shuffle.partitions", parameters["spark"]["shuffle_partitions"])
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)


Spark: 4.2.0


## 1. Load the final combined model population

Use the already-generated train / validation / OOT parquet outputs.

The audit population is the union of those three datasets. **All vintages are audited together.**


In [67]:
MODEL_SPLIT_ROOT = PROJECT_ROOT / "data" / "04_model_split" / APPROACH

TRAIN_PATH = MODEL_SPLIT_ROOT / "train_split.parquet"
VALIDATION_PATH = MODEL_SPLIT_ROOT / "validation_split.parquet"
OOT_PATH = MODEL_SPLIT_ROOT / "oot_split.parquet"

def load_required(path: Path) -> DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Expected parquet not found: {path}")
    return spark.read.parquet(str(path))

train_df = load_required(TRAIN_PATH)
validation_df = load_required(VALIDATION_PATH)
oot_df = load_required(OOT_PATH)

audit_df = (
    train_df
    .unionByName(validation_df, allowMissingColumns=True)
    .unionByName(oot_df, allowMissingColumns=True)
)

print("TRAIN rows:", train_df.count())
print("VALIDATION rows:", validation_df.count())
print("OOT rows:", oot_df.count())
print("COMBINED rows:", audit_df.count())
print("Columns:", len(audit_df.columns))


TRAIN rows: 10970334
VALIDATION rows: 10195946
OOT rows: 10897586
COMBINED rows: 32063866
Columns: 46


In [5]:
required_columns = ["loan_id", "period", TARGET, "vintage"] + MODEL_FEATURES
missing_columns = sorted(set(required_columns) - set(audit_df.columns))

print("Missing required audit/model columns:", missing_columns)

assert not missing_columns, (
    "Required model/audit columns are missing."
)

observed_vintages = sorted(
    r["vintage"]
    for r in audit_df.select("vintage").distinct().collect()
)

print("Observed vintages:", observed_vintages)
print("Configured vintages:", sorted(CONFIGURED_ALL_VINTAGES))

vintage_config_mismatch = (
    set(observed_vintages) != set(CONFIGURED_ALL_VINTAGES)
)

if vintage_config_mismatch:
    print(
        "REVIEW: configuration vintage list does not match observed model data. "
        "The audit will use OBSERVED vintages as the source of truth."
    )


Missing required audit/model columns: []
Observed vintages: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Configured vintages: [2015, 2016]
REVIEW: configuration vintage list does not match observed model data. The audit will use OBSERVED vintages as the source of truth.


## 2. Feature inventory and schema

The current behavioral model contains 23 configured predictors. The inventory below verifies their presence and Spark type.


In [6]:
schema_map = {
    field.name: field.dataType.simpleString()
    for field in audit_df.schema.fields
}

inventory_rows = []

for feature in NUMERIC_FEATURES:
    inventory_rows.append({
        "feature": feature,
        "class": "numeric",
        "spark_type": schema_map.get(feature),
        "present": feature in audit_df.columns,
    })

for feature in CATEGORICAL_FEATURES:
    inventory_rows.append({
        "feature": feature,
        "class": "categorical",
        "spark_type": schema_map.get(feature),
        "present": feature in audit_df.columns,
    })

for feature in ENGINEERED_FEATURES:
    inventory_rows.append({
        "feature": feature,
        "class": "engineered",
        "spark_type": schema_map.get(feature),
        "present": feature in audit_df.columns,
    })

inventory = pd.DataFrame(inventory_rows)
display(inventory)


,feature,class,spark_type,present
0,number_of_borrowers,numeric,bigint,True
1,mi_percentage,numeric,bigint,True
2,original_upb,numeric,bigint,True
3,current_actual_upb,numeric,double,True
4,current_interest_rate,numeric,double,True
5,estimated_ltv,numeric,bigint,True
6,calculated_loan_age,numeric,int,True
7,remaining_months_to_legal_maturity,numeric,bigint,True
8,current_dpd_numeric,numeric,double,True
9,max_dpd_to_date,numeric,double,True


## 3. Field-specific SFLLD domain contract

Do **not** use a generic 0–100 / 1–999 range for Freddie Mac fields.

The audit treats special codes as field-specific semantics:

- `original_dti`: 999 = Not Available
- `original_ltv`: 999 = Not Available; documented range varies historically
- `original_cltv`: 999 = Not Available; documented range varies historically
- `mi_percentage`: 0 = No MI; 1–55 = reported MI; 999 = Not Available
- `estimated_ltv`: 1–998 = usable; 999 = Unknown; NULL = Data Not Available
- categorical special codes are validated separately.

The model-input dataset has already undergone the documented origination sentinel normalization for DTI/LTV/CLTV/FTHB, so those checks are performed in two forms:

1. confirm no documented origination sentinels survive;
2. validate the remaining usable values.


In [7]:
# Documented final-model-input sentinel checks.
# These should be zero after the current origination normalization stage.

FINAL_MODEL_SENTINELS = {
    "original_dti": 999,
    "original_ltv": 999,
    "original_cltv": 999,
    "first_time_homebuyer_flag": "9",
}

sentinel_results = []

for feature, sentinel in FINAL_MODEL_SENTINELS.items():
    if feature not in audit_df.columns:
        continue

    count = (
        audit_df
        .filter(F.col(feature).cast("string") == F.lit(str(sentinel)))
        .count()
    )

    null_count = audit_df.filter(F.col(feature).isNull()).count()

    sentinel_results.append({
        "feature": feature,
        "documented_sentinel": sentinel,
        "remaining_count": count,
        "null_count": null_count,
        "status": "PASS" if count == 0 else "FAIL",
    })

sentinel_results = pd.DataFrame(sentinel_results)
display(sentinel_results)


,feature,documented_sentinel,remaining_count,null_count,status
0,original_dti,999,0,518276,PASS
1,original_ltv,999,0,187,PASS
2,original_cltv,999,0,368,PASS
3,first_time_homebuyer_flag,9,0,56,PASS


In [46]:
# Usable-domain checks.

checks = []

def add_check(name, condition, detail):
    count = audit_df.filter(condition).limit(1).count()
    checks.append({
        "check": name,
        "invalid_present": bool(count),
        "detail": detail,
    })
    return count

# DTI: usable values must be 0..65; null is permitted after sentinel normalization.
add_check(
    "original_dti_domain",
    F.col("original_dti").isNotNull()
    & ((F.col("original_dti") < 0) | (F.col("original_dti") > 65)),
    "Usable original DTI must be within 0..65; 999 is normalized away.",
)

# MI: 0, 1..55 are usable; 999 is not expected in final model input.
add_check(
    "mi_percentage_domain",
    F.col("mi_percentage").isNotNull()
    & (F.col("mi_percentage") != 999)
    & (F.col("mi_percentage") != 0)
    & ((F.col("mi_percentage") < 1) | (F.col("mi_percentage") > 55)),
    "MI must be 0 or 1..55 after excluding documented 999.",
)

# ELTV: 1..998 usable, 999 = Unknown.
add_check(
    "estimated_ltv_domain",
    F.col("estimated_ltv").isNotNull()
    & (F.col("estimated_ltv") != 999)
    & ((F.col("estimated_ltv") < 1) | (F.col("estimated_ltv") > 998)),
    "Usable ELTV must be 1..998; 999 is the documented Unknown code.",
)

# Generic non-negative sanity for values that are structurally non-negative.
for feature in [
    "original_upb",
    "current_actual_upb",
    "current_interest_rate",
    "original_interest_rate",
    "delinquency_months_to_date",
    "max_dpd_to_date",
    "current_dpd_numeric",
]:
    if feature in audit_df.columns:
        add_check(
            f"{feature}_non_negative",
            F.col(feature).isNotNull() & (F.col(feature) < 0),
            "Expected non-negative usable values.",
        )

domain_results = pd.DataFrame(checks)
display(domain_results)


,check,invalid_present,detail
0,original_dti_domain,False,Usable original DTI must be within 0..65; 999 ...
1,mi_percentage_domain,False,MI must be 0 or 1..55 after excluding document...
2,estimated_ltv_domain,False,Usable ELTV must be 1..998; 999 is the documen...
3,original_upb_non_negative,False,Expected non-negative usable values.
4,current_actual_upb_non_negative,False,Expected non-negative usable values.
5,current_interest_rate_non_negative,False,Expected non-negative usable values.
6,original_interest_rate_non_negative,False,Expected non-negative usable values.
7,delinquency_months_to_date_non_negative,False,Expected non-negative usable values.
8,max_dpd_to_date_non_negative,False,Expected non-negative usable values.
9,current_dpd_numeric_non_negative,False,Expected non-negative usable values.


## 4. Original LTV / CLTV historical ranges

Freddie Mac's documented ranges differ before versus from 2018Q2 onward.

For the current model dataset we do not need to guess the origination quarter from the observation period. Instead, use the loan's vintage and treat this as a **screening check**, while keeping any exact quarterly boundary confirmation as REVIEW.

The important checks are:

- 2015–2017 populations should satisfy the pre-2018Q2 range when usable.
- 2018+ populations should satisfy the post-2018Q2 range when usable.
- CLTV must not be below LTV when both are usable.


In [21]:
# ---------------------------------------------------------------
# HARP-aware LTV / CLTV validation
# ---------------------------------------------------------------

vintage_col = F.col("vintage").cast("int")
harp_col = F.upper(F.trim(F.col("harp_indicator").cast("string")))

# ---------------------------------------------------------------
# Original LTV
#
# Pre-2018Q2:
#   Non-HARP: 6..105
#   HARP:     1..998
#
# 2018Q2+:
#   1..998
#
# 999 is documented as Not Available and should not survive
# final-model preprocessing.
# ---------------------------------------------------------------

ltv_invalid_pre_non_harp = (
    audit_df.filter(
        (vintage_col <= F.lit(2017))
        & (harp_col.isNull() | (harp_col != F.lit("Y")))
        & F.col("original_ltv").isNotNull()
        & (F.col("original_ltv") != F.lit(999))
        & ((F.col("original_ltv") < F.lit(6)) | (F.col("original_ltv") > F.lit(105)))
    )
    .limit(1)
    .count()
)

ltv_invalid_pre_harp = (
    audit_df.filter(
        (vintage_col <= F.lit(2017))
        & (harp_col == F.lit("Y"))
        & F.col("original_ltv").isNotNull()
        & (F.col("original_ltv") != F.lit(999))
        & ((F.col("original_ltv") < F.lit(1)) | (F.col("original_ltv") > F.lit(998)))
    )
    .limit(1)
    .count()
)

ltv_invalid_post = (
    audit_df.filter(
        (vintage_col >= F.lit(2018))
        & F.col("original_ltv").isNotNull()
        & (F.col("original_ltv") != F.lit(999))
        & ((F.col("original_ltv") < F.lit(1)) | (F.col("original_ltv") > F.lit(998)))
    )
    .limit(1)
    .count()
)

# ---------------------------------------------------------------
# Original CLTV
#
# Pre-2018Q2:
#   Non-HARP: 6..200
#   HARP:     1..998
#
# 2018Q2+:
#   1..998
# ---------------------------------------------------------------

cltv_invalid_pre_non_harp = (
    audit_df.filter(
        (vintage_col <= F.lit(2017))
        & (harp_col.isNull() | (harp_col != F.lit("Y")))
        & F.col("original_cltv").isNotNull()
        & (F.col("original_cltv") != F.lit(999))
        & ((F.col("original_cltv") < F.lit(6)) | (F.col("original_cltv") > F.lit(200)))
    )
    .limit(1)
    .count()
)

cltv_invalid_pre_harp = (
    audit_df.filter(
        (vintage_col <= F.lit(2017))
        & (harp_col == F.lit("Y"))
        & F.col("original_cltv").isNotNull()
        & (F.col("original_cltv") != F.lit(999))
        & ((F.col("original_cltv") < F.lit(1)) | (F.col("original_cltv") > F.lit(998)))
    )
    .limit(1)
    .count()
)

cltv_invalid_post = (
    audit_df.filter(
        (vintage_col >= F.lit(2018))
        & F.col("original_cltv").isNotNull()
        & (F.col("original_cltv") != F.lit(999))
        & ((F.col("original_cltv") < F.lit(1)) | (F.col("original_cltv") > F.lit(998)))
    )
    .limit(1)
    .count()
)

# ---------------------------------------------------------------
# CLTV >= LTV
# ---------------------------------------------------------------

cltv_below_ltv = (
    audit_df.filter(
        F.col("original_cltv").isNotNull()
        & F.col("original_ltv").isNotNull()
        & (F.col("original_cltv") != F.lit(999))
        & (F.col("original_ltv") != F.lit(999))
        & (F.col("original_cltv") < F.col("original_ltv"))
    )
    .limit(1)
    .count()
)

# ---------------------------------------------------------------
# Results
# ---------------------------------------------------------------

ltv_cltv_results = pd.DataFrame(
    [
        {
            "check": "original_ltv_pre_2018_non_harp",
            "invalid_present": bool(ltv_invalid_pre_non_harp),
        },
        {
            "check": "original_ltv_pre_2018_harp",
            "invalid_present": bool(ltv_invalid_pre_harp),
        },
        {
            "check": "original_ltv_2018_plus",
            "invalid_present": bool(ltv_invalid_post),
        },
        {
            "check": "original_cltv_pre_2018_non_harp",
            "invalid_present": bool(cltv_invalid_pre_non_harp),
        },
        {
            "check": "original_cltv_pre_2018_harp",
            "invalid_present": bool(cltv_invalid_pre_harp),
        },
        {
            "check": "original_cltv_2018_plus",
            "invalid_present": bool(cltv_invalid_post),
        },
        {
            "check": "original_cltv_ge_original_ltv",
            "invalid_present": bool(cltv_below_ltv),
        },
    ]
)

display(ltv_cltv_results)

,check,invalid_present
0,original_ltv_pre_2018_non_harp,False
1,original_ltv_pre_2018_harp,False
2,original_ltv_2018_plus,False
3,original_cltv_pre_2018_non_harp,False
4,original_cltv_pre_2018_harp,False
5,original_cltv_2018_plus,False
6,original_cltv_ge_original_ltv,False


In [10]:
pre_2018_violations = (
    audit_df.filter(
        (F.col("vintage") <= F.lit(2017))
        & F.col("original_ltv").isNotNull()
        & ((F.col("original_ltv") < F.lit(6)) | (F.col("original_ltv") > F.lit(105)))
    )
    .groupBy("vintage", "original_ltv")
    .count()
    .orderBy("vintage", "original_ltv")
)

pre_2018_violations.show()

+-------+------------+-----+
|vintage|original_ltv|count|
+-------+------------+-----+
|   2015|           1|    2|
|   2015|           2|    2|
|   2015|           3|   20|
|   2015|           4|   34|
|   2015|           5|   52|
|   2015|         106| 1090|
|   2015|         107| 1098|
|   2015|         108| 1057|
|   2015|         109|  930|
|   2015|         110|  872|
|   2015|         111|  865|
|   2015|         112|  876|
|   2015|         113|  701|
|   2015|         114|  712|
|   2015|         115|  667|
|   2015|         116|  630|
|   2015|         117|  559|
|   2015|         118|  525|
|   2015|         119|  515|
|   2015|         120|  447|
+-------+------------+-----+
only showing top 20 rows


In [12]:
ltv_violations_by_harp = (
    audit_df.filter(
        (F.col("vintage") <= 2017)
        & F.col("original_ltv").isNotNull()
        & ((F.col("original_ltv") < 6) | (F.col("original_ltv") > 105))
    )
    .groupBy(
        "vintage",
        "harp_indicator",
    )
    .agg(
        F.count("*").alias("rows"),
        F.min("original_ltv").alias("min_ltv"),
        F.max("original_ltv").alias("max_ltv"),
    )
    .orderBy("vintage", "harp_indicator")
)

ltv_violations_by_harp.show()

+-------+--------------+-----+-------+-------+
|vintage|harp_indicator| rows|min_ltv|max_ltv|
+-------+--------------+-----+-------+-------+
|   2015|             Y|20519|      1|    929|
|   2016|             Y|10388|      2|    663|
|   2017|             Y| 4903|      2|    611|
+-------+--------------+-----+-------+-------+



In [18]:
(
    audit_df.filter(
        (F.col("vintage") <= 2017)
        & F.col("original_ltv").isNotNull()
        & ((F.col("original_ltv") < 6) | (F.col("original_ltv") > 105))
    )
    .groupBy("harp_indicator")
    .agg(
        F.count("*").alias("rows"),
        F.min("original_ltv").alias("min_ltv"),
        F.max("original_ltv").alias("max_ltv"),
    )
    .orderBy("harp_indicator")
).show()

+--------------+-----+-------+-------+
|harp_indicator| rows|min_ltv|max_ltv|
+--------------+-----+-------+-------+
|             Y|35810|      1|    929|
+--------------+-----+-------+-------+



In [20]:
cltv_violations_by_harp = (
    audit_df.filter(
        (F.col("vintage") <= 2017)
        & F.col("original_cltv").isNotNull()
        & ((F.col("original_cltv") < 6) | (F.col("original_cltv") > 200))
    )
    .groupBy("vintage", "harp_indicator")
    .agg(
        F.count("*").alias("rows"),
        F.min("original_cltv").alias("min_cltv"),
        F.max("original_cltv").alias("max_cltv"),
    )
    .orderBy("vintage", "harp_indicator")
)

cltv_violations_by_harp.show()

+-------+--------------+----+--------+--------+
|vintage|harp_indicator|rows|min_cltv|max_cltv|
+-------+--------------+----+--------+--------+
|   2015|             Y| 723|       1|     931|
|   2016|             Y| 334|       3|     899|
|   2017|             Y| 154|       2|     645|
+-------+--------------+----+--------+--------+



## 5. ELTV temporal availability

Freddie Mac documents monthly estimated LTV as available beginning in **April 2017**.

The final behavioral dataset stores `period` in the pipeline's canonical monthly representation. This notebook does **not** hard-code an assumed encoding.

It first attempts to infer whether `period` is:

- `YYYYMM`, or
- a monthly ordinal.

If the encoding cannot be determined confidently, the check is marked **REVIEW** rather than producing a false FAIL.


In [24]:
def infer_period_encoding(df: DataFrame, column="period"):
    sample = (
        df.select(F.col(column).cast("long").alias("period"))
        .where(F.col(column).isNotNull())
        .agg(
            F.min("period").alias("min_period"),
            F.max("period").alias("max_period"),
        )
        .first()
    )

    min_period = sample["min_period"]
    max_period = sample["max_period"]

    if min_period is None:
        return {"encoding": None, "reason": "No non-null period values."}

    # YYYYMM encoding.
    if 190001 <= min_period <= 210012 and 1 <= max_period % 100 <= 12:
        return {
            "encoding": "YYYYMM",
            "min_period": min_period,
            "max_period": max_period,
        }

    # Common monthly ordinal / integer encoding.
    return {
        "encoding": "unknown_integer",
        "min_period": min_period,
        "max_period": max_period,
    }

period_encoding = infer_period_encoding(audit_df)

print(period_encoding)


{'encoding': 'unknown_integer', 'min_period': 546, 'max_period': 662}


In [31]:
ELTV_START_PERIOD = 567

eltv_pre_availability_invalid = (
    audit_df.filter(
        (F.col("period") < F.lit(ELTV_START_PERIOD))
        & F.col("estimated_ltv").isNotNull()
        & (F.col("estimated_ltv") != F.lit(999))
    )
    .limit(1)
    .count()
)

eltv_temporal_status = "PASS" if eltv_pre_availability_invalid == 0 else "FAIL"

eltv_temporal_detail = (
    "Before April 2017, all non-null ELTV values are the documented "
    "999/Unknown code; no actual ELTV ratios are populated."
    if eltv_pre_availability_invalid == 0
    else "Actual ELTV ratios (1-998) are populated before April 2017."
)

pd.DataFrame(
    [
        {
            "check": "estimated_ltv_temporal_availability",
            "status": eltv_temporal_status,
            "invalid_rows_present": bool(eltv_pre_availability_invalid),
            "cutoff_period": ELTV_START_PERIOD,
            "cutoff_date": "2017-04",
            "detail": eltv_temporal_detail,
        }
    ]
)

,check,status,invalid_rows_present,cutoff_period,cutoff_date,detail
0,estimated_ltv_temporal_availability,PASS,False,567,2017-04,"Before April 2017, all non-null ELTV values ar..."


In [27]:
eltv_pre_availability = (
    audit_df.filter((F.col("period") < F.lit(567)) & F.col("estimated_ltv").isNotNull())
    .groupBy("period", "estimated_ltv")
    .count()
    .orderBy("period", "estimated_ltv")
)

eltv_pre_availability.show()

+------+-------------+------+
|period|estimated_ltv| count|
+------+-------------+------+
|   546|          999|   940|
|   547|          999| 87256|
|   548|          999|114014|
|   549|          999|146822|
|   550|          999|136088|
|   551|          999|129359|
|   552|          999|128303|
|   553|          999|203243|
|   554|          999|223896|
|   555|          999|254167|
|   556|          999|241428|
|   557|          999|221934|
|   558|          999|229776|
|   559|          999|196201|
|   560|          999|194436|
|   561|          999|226104|
|   562|          999|229872|
|   563|          999|228262|
|   564|          999|254059|
|   565|          999|224306|
+------+-------------+------+
only showing top 20 rows


In [30]:
(
    audit_df.filter((F.col("period") < F.lit(567)) & F.col("estimated_ltv").isNotNull())
    .select("period")
    .distinct()
    .withColumn(
        "period_date",
        F.add_months(
            F.to_date(F.lit("1970-01-01")),
            F.col("period").cast("int"),
        ),
    )
    .orderBy("period").show()
)

+------+-----------+
|period|period_date|
+------+-----------+
|   546| 2015-07-01|
|   547| 2015-08-01|
|   548| 2015-09-01|
|   549| 2015-10-01|
|   550| 2015-11-01|
|   551| 2015-12-01|
|   552| 2016-01-01|
|   553| 2016-02-01|
|   554| 2016-03-01|
|   555| 2016-04-01|
|   556| 2016-05-01|
|   557| 2016-06-01|
|   558| 2016-07-01|
|   559| 2016-08-01|
|   560| 2016-09-01|
|   561| 2016-10-01|
|   562| 2016-11-01|
|   563| 2016-12-01|
|   564| 2017-01-01|
|   565| 2017-02-01|
+------+-----------+
only showing top 20 rows


## 6. Categorical domain checks

Validate documented SFLLD category codes rather than assuming that any Spark string value is legitimate.


In [32]:
CATEGORICAL_DOMAINS = {
    "first_time_homebuyer_flag": {"Y", "N", "9"},
    "occupancy_status": {"P", "I", "S", "9"},
    "property_type": {"CO", "PU", "MH", "SF", "CP", "99"},
    "loan_purpose": {"P", "C", "N", "R", "9"},
    "channel": {"R", "B", "C", "T", "9"},
}

categorical_results = []

for feature, valid_values in CATEGORICAL_DOMAINS.items():
    if feature not in audit_df.columns:
        continue

    invalid = (
        audit_df
        .filter(
            F.col(feature).isNotNull()
            & ~F.col(feature).isin(*sorted(valid_values))
        )
        .limit(1)
        .count()
    )

    categorical_results.append({
        "feature": feature,
        "invalid_present": bool(invalid),
        "valid_domain": sorted(valid_values),
    })

display(pd.DataFrame(categorical_results))


,feature,invalid_present,valid_domain
0,first_time_homebuyer_flag,False,"[9, N, Y]"
1,occupancy_status,False,"[9, I, P, S]"
2,property_type,False,"[99, CO, CP, MH, PU, SF]"
3,loan_purpose,False,"[9, C, N, P, R]"
4,channel,False,"[9, B, C, R, T]"


In [33]:
# Explicitly audit the categorical columns not covered by the compact domain map.

# Number of borrowers has historical encoding changes; validate the legal
# code set separately.
borrower_invalid = (
    audit_df
    .filter(
        F.col("number_of_borrowers").isNotNull()
        & ~F.col("number_of_borrowers").isin(
            "01", "02", "03", "04", "05",
            "06", "07", "08", "09", "10", "99",
            1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 99,
        )
    )
    .limit(1)
    .count()
)

state_invalid_length = (
    audit_df
    .filter(
        F.col("property_state").isNotNull()
        & (F.length(F.col("property_state")) != 2)
    )
    .limit(1)
    .count()
)

display(pd.DataFrame([
    {
        "check": "number_of_borrowers_code_set",
        "invalid_present": bool(borrower_invalid),
    },
    {
        "check": "property_state_two_character_code",
        "invalid_present": bool(state_invalid_length),
    },
]))


,check,invalid_present
0,number_of_borrowers_code_set,False
1,property_state_two_character_code,False


## 7. Missingness semantics

Missingness is a data-semantic audit, not a blanket instruction to impute.

One particularly important current field is:

`months_since_last_delinquency`

A NULL value can mean the borrower has **no prior delinquency**, depending on the upstream feature-construction contract.

We therefore inspect its raw distribution and retain that semantic distinction for modelling.


In [34]:
missing_rows = []

for feature in MODEL_FEATURES:
    row = audit_df.select(
        F.count("*").alias("rows"),
        F.sum(F.when(F.col(feature).isNull(), 1).otherwise(0)).alias("nulls"),
    ).first()

    rows = int(row["rows"])
    nulls = int(row["nulls"] or 0)

    missing_rows.append({
        "feature": feature,
        "rows": rows,
        "nulls": nulls,
        "null_pct": nulls / rows,
    })

missing_summary = (
    pd.DataFrame(missing_rows)
    .sort_values("null_pct", ascending=False)
)

display(missing_summary)


,feature,rows,nulls,null_pct
11,months_since_last_delinquency,32063866,31199790,0.973051
14,first_time_homebuyer_flag,32063866,56,0.000002
0,number_of_borrowers,32063866,0,0.000000
3,current_actual_upb,32063866,0,0.000000
4,current_interest_rate,32063866,0,0.000000
1,mi_percentage,32063866,0,0.000000
2,original_upb,32063866,0,0.000000
6,calculated_loan_age,32063866,0,0.000000
5,estimated_ltv,32063866,0,0.000000
7,remaining_months_to_legal_maturity,32063866,0,0.000000


In [35]:
# Current production modelling semantics for the two special cases we know about.

semantic_checks = []

if "original_dti_missing" in audit_df.columns:
    dti_indicator_mismatch = audit_df.filter(
        (
            F.when(F.col("original_dti").isNull(), F.lit(1))
            .otherwise(F.lit(0))
        )
        != F.col("original_dti_missing")
    ).limit(1).count()

    semantic_checks.append({
        "check": "original_dti_missing_matches_original_dti",
        "invalid_present": bool(dti_indicator_mismatch),
    })

if "months_since_last_delinquency" in audit_df.columns:
    null_count = audit_df.filter(
        F.col("months_since_last_delinquency").isNull()
    ).count()

    semantic_checks.append({
        "check": "months_since_last_delinquency_nulls",
        "invalid_present": False,
        "detail": (
            f"{null_count:,} nulls observed; null is retained as a distinct "
            "state pending the feature-construction contract."
        ),
    })

display(pd.DataFrame(semantic_checks))


,check,invalid_present,detail
0,original_dti_missing_matches_original_dti,False,NaN
1,months_since_last_delinquency_nulls,False,"31,199,790 nulls observed; null is retained as..."


## 8. Derived-feature consistency

Check identities that can be verified from columns actually present in the final model dataset.

We do **not** refer to `maturity_date` or `first_payment_date` here because those fields are not present in the final audit population.


In [36]:
derived_checks = []

# UPB change identity.
if {
    "upb_change_from_origination",
    "current_actual_upb",
    "original_upb",
}.issubset(audit_df.columns):
    upb_violation = audit_df.filter(
        F.col("upb_change_from_origination").isNotNull()
        & F.col("current_actual_upb").isNotNull()
        & F.col("original_upb").isNotNull()
        & (
            F.abs(
                F.col("upb_change_from_origination")
                - (
                    F.col("current_actual_upb")
                    - F.col("original_upb")
                )
            ) > F.lit(1e-6)
        )
    ).limit(1).count()

    derived_checks.append({
        "check": "upb_change_identity",
        "invalid_present": bool(upb_violation),
    })

# Percentage-change identity.
if {
    "upb_pct_change_from_origination",
    "upb_change_from_origination",
    "original_upb",
}.issubset(audit_df.columns):
    pct_violation = audit_df.filter(
        F.col("upb_pct_change_from_origination").isNotNull()
        & F.col("upb_change_from_origination").isNotNull()
        & F.col("original_upb").isNotNull()
        & (F.col("original_upb") != 0)
        & (
            F.abs(
                F.col("upb_pct_change_from_origination")
                - (
                    F.col("upb_change_from_origination")
                    / F.col("original_upb")
                )
            ) > F.lit(1e-6)
        )
    ).limit(1).count()

    derived_checks.append({
        "check": "upb_pct_change_identity",
        "invalid_present": bool(pct_violation),
    })

# Lifetime maximum cannot be below current DPD.
if {"max_dpd_to_date", "current_dpd_numeric"}.issubset(audit_df.columns):
    dpd_violation = audit_df.filter(
        F.col("max_dpd_to_date").isNotNull()
        & F.col("current_dpd_numeric").isNotNull()
        & (F.col("max_dpd_to_date") < F.col("current_dpd_numeric"))
    ).limit(1).count()

    derived_checks.append({
        "check": "max_dpd_ge_current_dpd",
        "invalid_present": bool(dpd_violation),
    })

# Loan-term lifecycle consistency using only final-dataset columns.
# This is an internal identity check, not a substitute for the upstream
# First Payment Date / Maturity Date source audit.
if {
    "original_loan_term",
    "calculated_loan_age",
    "remaining_months_to_legal_maturity",
}.issubset(audit_df.columns):
    term_violation = audit_df.filter(
        F.col("original_loan_term").isNotNull()
        & F.col("calculated_loan_age").isNotNull()
        & F.col("remaining_months_to_legal_maturity").isNotNull()
        & (
            F.col("original_loan_term")
            != (
                F.col("calculated_loan_age")
                + F.col("remaining_months_to_legal_maturity")
                - F.lit(1)
            )
        )
    ).limit(1).count()

    derived_checks.append({
        "check": "loan_term_lifecycle_identity",
        "invalid_present": bool(term_violation),
    })

display(pd.DataFrame(derived_checks))


,check,invalid_present
0,upb_change_identity,False
1,upb_pct_change_identity,False
2,max_dpd_ge_current_dpd,False
3,loan_term_lifecycle_identity,True


In [39]:
loan_term_failures = (
    audit_df.filter(
        F.col("original_loan_term").isNotNull()
        & F.col("calculated_loan_age").isNotNull()
        & F.col("remaining_months_to_legal_maturity").isNotNull()
        & (
            F.col("original_loan_term")
            != (
                F.col("calculated_loan_age")
                + F.col("remaining_months_to_legal_maturity")
                - F.lit(1)
            )
        )
    )
    .select(
        "loan_id",
        "vintage",
        "period",
        "original_loan_term",
        "calculated_loan_age",
        "remaining_months_to_legal_maturity",
    )
    .withColumn(
        "expected_term",
        F.col("calculated_loan_age")
        + F.col("remaining_months_to_legal_maturity")
        - F.lit(1),
    )
    .withColumn(
        "difference",
        F.col("original_loan_term") - F.col("expected_term"),
    )
)

display(loan_term_failures.groupBy("difference").count().orderBy("difference").show())

+----------+-----+
|difference|count|
+----------+-----+
|      -330|    2|
|      -310|    1|
|      -228|    2|
|      -226|    2|
|      -225|    2|
|      -224|    2|
|      -223|    2|
|      -220|    2|
|      -217|    3|
|      -196|    1|
|      -195|    1|
|      -190|    2|
|      -189|    2|
|      -187|    3|
|      -186|    2|
|      -185|    2|
|      -184|    2|
|      -182|    1|
|      -181|    2|
|      -180|    5|
+----------+-----+
only showing top 20 rows


None

In [41]:
display(
    loan_term_failures.select(
        "loan_id",
        "vintage",
        "period",
        "original_loan_term",
        "calculated_loan_age",
        "remaining_months_to_legal_maturity",
        "expected_term",
        "difference",
    )
    .orderBy(F.abs(F.col("difference")).desc())
    .limit(30).show()
)

+------------+-------+------+------------------+-------------------+----------------------------------+-------------+----------+
|     loan_id|vintage|period|original_loan_term|calculated_loan_age|remaining_months_to_legal_maturity|expected_term|difference|
+------------+-------+------+------------------+-------------------+----------------------------------+-------------+----------+
|F15Q10082540|   2015|   548|               180|                  6|                               505|          510|      -330|
|F15Q10082540|   2015|   554|               180|                 12|                               499|          510|      -330|
|F17Q10278915|   2017|   579|               180|                 12|                               479|          490|      -310|
|F16Q40065249|   2016|   568|               360|                  6|                               583|          588|      -228|
|F16Q40065249|   2016|   574|               360|                 12|                             

None

In [43]:
display(
    audit_df.select(
        "original_loan_term",
        "calculated_loan_age",
        "remaining_months_to_legal_maturity",
    ).summary().show()
)

+-------+------------------+-------------------+----------------------------------+
|summary|original_loan_term|calculated_loan_age|remaining_months_to_legal_maturity|
+-------+------------------+-------------------+----------------------------------+
|  count|          32063866|           32063866|                          32063866|
|   mean| 322.0664797563712|  8.893723919629654|                313.17387622565536|
| stddev| 71.22210888186831| 2.9981170215784823|                 71.29676363940504|
|    min|                60|                  6|                                48|
|    25%|               360|                  6|                               348|
|    50%|               360|                  6|                               348|
|    75%|               360|                 12|                               354|
|    max|               559|                 12|                               583|
+-------+------------------+-------------------+----------------------------

None

In [44]:
display(
    audit_df.select(
        "original_loan_term",
        "calculated_loan_age",
        "remaining_months_to_legal_maturity",
        (
            F.col("original_loan_term")
            - F.col("calculated_loan_age")
            - F.col("remaining_months_to_legal_maturity")
        ).alias("term_residual"),
    ).summary().show()
)

+-------+------------------+-------------------+----------------------------------+--------------------+
|summary|original_loan_term|calculated_loan_age|remaining_months_to_legal_maturity|       term_residual|
+-------+------------------+-------------------+----------------------------------+--------------------+
|  count|          32063866|           32063866|                          32063866|            32063866|
|   mean| 322.0664797563712|  8.893723919629654|                313.17387622565536|-0.00112038891380...|
| stddev| 71.22210888186831| 2.9981170215784823|                 71.29676363940504|  0.4328730611331799|
|    min|                60|                  6|                                48|                -331|
|    25%|               360|                  6|                               348|                   0|
|    50%|               360|                  6|                               348|                   0|
|    75%|               360|                 12|       

None

In [47]:
expected_remaining = F.col("original_loan_term") - F.col("calculated_loan_age")

loan_term_invalid = (
    audit_df.filter(
        F.col("original_loan_term").isNotNull()
        & F.col("calculated_loan_age").isNotNull()
        & F.col("remaining_months_to_legal_maturity").isNotNull()
        & (F.col("remaining_months_to_legal_maturity") != expected_remaining)
    )
    .limit(1)
    .count()
)

In [49]:
loan_term_invalid

1

## 9. Grain and target sanity

The behavioral modelling grain is one row per:

`loan_id + period`

The target must be non-null and the observation-age values must be the configured 6M / 12M observations.


In [37]:
duplicate_groups = (
    audit_df
    .groupBy("loan_id", "period")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_group_count = duplicate_groups.count()

target_null_count = audit_df.filter(F.col(TARGET).isNull()).limit(1).count()

if "observation_age" in audit_df.columns:
    unexpected_observation_age = (
        audit_df
        .filter(~F.col("observation_age").isin(6, 12))
        .limit(1)
        .count()
    )
else:
    unexpected_observation_age = None

print("Duplicate (loan_id, period) groups:", duplicate_group_count)
print("Target null present:", bool(target_null_count))
print("Unexpected observation age present:", unexpected_observation_age)


Duplicate (loan_id, period) groups: 0
Target null present: False
Unexpected observation age present: 0


## 10. Cross-vintage sanity

Cross-vintage checks are diagnostic. A distribution shift is not automatically a correctness failure.

We flag values that indicate obvious semantic problems, while leaving legitimate population shifts for the separate V3 stability analysis.


In [46]:
vintage_population = (
    audit_df
    .groupBy("vintage")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("loan_id").alias("loans"),
    )
    .orderBy("vintage")
    .toPandas()
)

display(vintage_population)


,vintage,rows,loans
0,2015,2741047,1405243
1,2016,3233622,1644829
2,2017,2623879,1335046
3,2018,2371786,1246134
4,2019,3063859,1696377
5,2020,7132087,3747960
6,2021,7846853,3993807
7,2022,3050733,1544883


In [50]:
# Compact min / median / max diagnostics for numerical model features.
# This is designed to expose impossible values or obvious coding issues.

numeric_cross_vintage = []

for feature in NUMERIC_FEATURES + ENGINEERED_FEATURES:
    if feature not in audit_df.columns:
        continue

    stats = (
        audit_df
        .groupBy("vintage")
        .agg(
            F.count(F.col(feature)).alias("non_null"),
            F.min(F.col(feature)).alias("min"),
            F.expr(
                f"percentile_approx(`{feature}`, 0.50, 10000)"
            ).alias("median"),
            F.max(F.col(feature)).alias("max"),
        )
        .withColumn("feature", F.lit(feature))
        .select(
            "feature", "vintage", "non_null", "min", "median", "max"
        )
    )

    numeric_cross_vintage.append(stats)

if numeric_cross_vintage:
    combined_numeric = numeric_cross_vintage[0]
    for frame in numeric_cross_vintage[1:]:
        combined_numeric = combined_numeric.unionByName(frame)

    display(
        combined_numeric
        .orderBy("feature", "vintage")
        .toPandas()
    )


,feature,vintage,non_null,min,median,max
0,calculated_loan_age,2015,2741047,6.0,6.000000,12.000000
1,calculated_loan_age,2016,3233622,6.0,6.000000,12.000000
2,calculated_loan_age,2017,2623879,6.0,6.000000,12.000000
3,calculated_loan_age,2018,2371786,6.0,6.000000,12.000000
4,calculated_loan_age,2019,3063859,6.0,6.000000,12.000000
...,...,...,...,...,...,...
115,upb_pct_change_from_origination,2018,2371786,-1.0,-0.014965,0.005359
116,upb_pct_change_from_origination,2019,3063859,-1.0,-0.016173,0.033333
117,upb_pct_change_from_origination,2020,7132087,-1.0,-0.020878,0.049296
118,upb_pct_change_from_origination,2021,7846853,-1.0,-0.020913,0.031336


## 11. Feature lineage register

This is the working semantic register for the current model. The register explicitly marks fields that require an upstream source audit.

The register is not claimed to replace the underlying Freddie Mac documentation.


In [51]:
FEATURE_LINEAGE = {
    "number_of_borrowers": (
        "Freddie Mac origination field",
        "raw",
        "origination",
        "Historical encoding changed around 2018Q2.",
    ),
    "mi_percentage": (
        "Freddie Mac origination field",
        "raw",
        "origination",
        "0=no MI; 1..55 reported MI; 999=N/A.",
    ),
    "original_upb": (
        "Freddie Mac origination field",
        "raw",
        "origination",
        "Original UPB.",
    ),
    "original_interest_rate": (
        "Freddie Mac origination field",
        "raw",
        "origination",
        "Original contract rate.",
    ),
    "original_loan_term": (
        "Freddie Mac origination field",
        "raw",
        "origination",
        "Upstream date-based validation should be retained.",
    ),
    "current_actual_upb": (
        "Freddie Mac monthly performance field",
        "raw",
        "point_in_time",
        "Current UPB at observation month.",
    ),
    "current_interest_rate": (
        "Freddie Mac monthly performance field",
        "raw",
        "point_in_time",
        "Current rate at observation month.",
    ),
    "estimated_ltv": (
        "Freddie Mac monthly performance field",
        "raw",
        "point_in_time",
        "1..998 usable; 999=Unknown; NULL=Data Not Available; availability starts Apr-2017.",
    ),
    "calculated_loan_age": (
        "period + first-payment timing",
        "derived",
        "point_in_time",
        "Must be point-in-time and start at configured observation ages.",
    ),
    "remaining_months_to_legal_maturity": (
        "period + maturity timing",
        "derived",
        "point_in_time",
        "Must be point-in-time.",
    ),
    "current_dpd_numeric": (
        "current_loan_delinquency_status",
        "derived",
        "point_in_time",
        "Non-numeric performance codes must not be interpreted as DPD.",
    ),
    "max_dpd_to_date": (
        "current_dpd_numeric history",
        "derived",
        "point_in_time",
        "Maximum through observation month only.",
    ),
    "delinquency_months_to_date": (
        "current_dpd_numeric history",
        "derived",
        "point_in_time",
        "Count through observation month only.",
    ),
    "months_since_last_delinquency": (
        "current_dpd_numeric history",
        "derived",
        "point_in_time",
        "No-prior-delinquency state must retain documented semantics.",
    ),
    "upb_change_from_origination": (
        "current_actual_upb + original_upb",
        "derived",
        "point_in_time",
        "Direct difference.",
    ),
    "upb_pct_change_from_origination": (
        "current_actual_upb + original_upb",
        "derived",
        "point_in_time",
        "Relative difference.",
    ),
}

lineage_rows = []

for feature in MODEL_FEATURES:
    source, transformation, availability, notes = FEATURE_LINEAGE.get(
        feature,
        ("TBD", "TBD", "TBD", "No register entry yet."),
    )
    lineage_rows.append({
        "feature": feature,
        "source": source,
        "transformation": transformation,
        "availability": availability,
        "notes": notes,
    })

lineage_df = pd.DataFrame(lineage_rows)
display(lineage_df)


,feature,source,transformation,availability,notes
0,number_of_borrowers,Freddie Mac origination field,raw,origination,Historical encoding changed around 2018Q2.
1,mi_percentage,Freddie Mac origination field,raw,origination,0=no MI; 1..55 reported MI; 999=N/A.
2,original_upb,Freddie Mac origination field,raw,origination,Original UPB.
3,current_actual_upb,Freddie Mac monthly performance field,raw,point_in_time,Current UPB at observation month.
4,current_interest_rate,Freddie Mac monthly performance field,raw,point_in_time,Current rate at observation month.
5,estimated_ltv,Freddie Mac monthly performance field,raw,point_in_time,1..998 usable; 999=Unknown; NULL=Data Not Avai...
6,calculated_loan_age,period + first-payment timing,derived,point_in_time,Must be point-in-time and start at configured ...
7,remaining_months_to_legal_maturity,period + maturity timing,derived,point_in_time,Must be point-in-time.
8,current_dpd_numeric,current_loan_delinquency_status,derived,point_in_time,Non-numeric performance codes must not be inte...
9,max_dpd_to_date,current_dpd_numeric history,derived,point_in_time,Maximum through observation month only.


## 12. Consolidated audit result

The final summary intentionally distinguishes:

- **FAIL** = an actual structural/domain violation was observed.
- **REVIEW** = the data check is valid but the current final dataset cannot prove the point without upstream/source documentation.
- **PASS** = the check is executable and passed.

A `REVIEW` item should be resolved before declaring the V3 correctness gate complete.


In [52]:
summary_rows = []

def add_summary(check, status, detail):
    summary_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

# Feature presence
add_summary(
    "configured_features_present",
    "PASS" if not missing_columns else "FAIL",
    "All configured model/audit columns are present."
    if not missing_columns else f"Missing: {missing_columns}",
)

# Final sentinels
if not sentinel_results.empty:
    sentinel_failures = int((sentinel_results["remaining_count"] > 0).sum())
else:
    sentinel_failures = 0

add_summary(
    "documented_origination_sentinels_removed",
    "PASS" if sentinel_failures == 0 else "FAIL",
    f"{sentinel_failures} sentinel fields retain documented sentinel codes.",
)

# Domain checks
domain_failures = int(domain_results["invalid_present"].sum()) if not domain_results.empty else 0
add_summary(
    "field_specific_numeric_domains",
    "PASS" if domain_failures == 0 else "FAIL",
    f"{domain_failures} executable numeric-domain checks found violations.",
)

ltv_cltv_failures = int(ltv_cltv_results["invalid_present"].sum())
add_summary(
    "ltv_cltv_sanity",
    "PASS" if ltv_cltv_failures == 0 else "FAIL",
    f"{ltv_cltv_failures} LTV/CLTV checks found violations.",
)

categorical_failures = int(categorical_results["invalid_present"].sum()) if not categorical_results.empty else 0
categorical_extra_failures = int(
    borrower_invalid + state_invalid_length
)

add_summary(
    "categorical_domain_validity",
    "PASS" if (categorical_failures + categorical_extra_failures) == 0 else "FAIL",
    "All audited categorical domains are valid.",
)

# Derived identities
derived_failures = int(derived_checks["invalid_present"].sum()) if not derived_checks.empty else 0
add_summary(
    "derived_feature_identities",
    "PASS" if derived_failures == 0 else "FAIL",
    f"{derived_failures} executable derived-feature identity checks found violations.",
)

# Grain / target
add_summary(
    "behavioral_grain_unique",
    "PASS" if duplicate_group_count == 0 else "FAIL",
    f"Duplicate (loan_id, period) groups: {duplicate_group_count}.",
)

add_summary(
    "target_complete",
    "PASS" if target_null_count == 0 else "FAIL",
    f"Target null present: {bool(target_null_count)}.",
)

if unexpected_observation_age is not None:
    add_summary(
        "observation_age_valid",
        "PASS" if unexpected_observation_age == 0 else "FAIL",
        "Only observation ages 6 and 12 are present.",
    )

# ELTV
add_summary(
    "estimated_ltv_temporal_availability",
    eltv_temporal_status,
    eltv_temporal_detail,
)

# Config-vintage mismatch is useful but not a feature-data failure.
add_summary(
    "configured_vs_observed_vintages",
    "REVIEW" if vintage_config_mismatch else "PASS",
    (
        f"Configured={sorted(CONFIGURED_ALL_VINTAGES)}, "
        f"observed={observed_vintages}. Audit uses observed vintages."
        if vintage_config_mismatch
        else "Configured and observed vintages agree."
    ),
)

# Documentation / source review.
add_summary(
    "sfll_d_field_definition_reconciliation",
    "REVIEW",
    "Maintain a field-by-field source-document crosswalk for the final feature set.",
)

add_summary(
    "upstream_loan_term_source_validation",
    "REVIEW",
    "Final model parquet does not contain First Payment Date/Maturity Date; "
    "upstream origination validation remains the authoritative source check.",
)

audit_summary = pd.DataFrame(summary_rows)
display(audit_summary)

print("\nCounts by status:")
print(audit_summary["status"].value_counts(dropna=False))


NameError: name 'domain_results' is not defined

# 13. V3 Feature Correctness Gate

### Mark #1 ✅ only when:

**FAIL count = 0**

and every **REVIEW** item has a documented resolution.

For the current dataset, the most important remaining reviews are expected to be:

- exact SFLLD field-definition reconciliation;
- upstream validation of `original_loan_term`;
- confirmation of the canonical `period` encoding for the ELTV April-2017 availability rule;
- any configuration-vs-observed vintage mismatch.

Do not treat correlation, feature importance, or model performance as evidence that a feature is semantically correct. Those belong to later V3 workstreams.


In [68]:
audit_df.columns

['loan_id',
 'observation_age',
 'period',
 'current_actual_upb',
 'current_interest_rate',
 'loan_age',
 'remaining_months_to_legal_maturity',
 'estimated_ltv',
 'current_loan_delinquency_status',
 'ddlpi',
 'zero_balance_code',
 'zero_balance_effective_date',
 'credit_score',
 'original_dti',
 'number_of_borrowers',
 'original_ltv',
 'original_cltv',
 'mi_percentage',
 'original_upb',
 'original_interest_rate',
 'original_loan_term',
 'first_time_homebuyer_flag',
 'property_type',
 'occupancy_status',
 'loan_purpose',
 'channel',
 'super_conforming_flag',
 'harp_indicator',
 'property_state',
 'original_dti_missing',
 'first_payment_date',
 'calculated_loan_age',
 'current_dpd_numeric',
 'current_dpd_30_plus',
 'current_dpd_60_plus',
 'current_delinquency_flag',
 'upb_change_from_origination',
 'upb_pct_change_from_origination',
 'rate_change_from_origination',
 'max_dpd_to_date',
 'ever_30dpd_to_date',
 'ever_60dpd_to_date',
 'delinquency_months_to_date',
 'months_since_last_delinqu

In [59]:
credit_score_summary = audit_df.select("credit_score").summary()

display(credit_score_summary.show())

+-------+------------------+
|summary|      credit_score|
+-------+------------------+
|  count|          32063866|
|   mean| 752.3017925224613|
| stddev|122.54469846798379|
|    min|               300|
|    25%|               720|
|    50%|               760|
|    75%|               788|
|    max|              9999|
+-------+------------------+



None

In [61]:
display(
    audit_df.agg(
        F.count("*").alias("rows"),
        F.sum(F.when(F.col("credit_score").isNull(), 1).otherwise(0)).alias("nulls"),
        F.sum(F.when(F.col("credit_score") == 9999, 1).otherwise(0)).alias(
            "sentinel_9999"
        ),
        F.min("credit_score").alias("min"),
        F.max("credit_score").alias("max"),
    ).show()
)

+--------+-----+-------------+---+----+
|    rows|nulls|sentinel_9999|min| max|
+--------+-----+-------------+---+----+
|32063866|    0|         4865|300|9999|
+--------+-----+-------------+---+----+



None

In [62]:
display(
    audit_df.groupBy("vintage")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.when(F.col("credit_score").isNull(), 1).otherwise(0)).alias("nulls"),
        F.sum(F.when(F.col("credit_score") == 9999, 1).otherwise(0)).alias(
            "sentinel_9999"
        ),
        F.min("credit_score").alias("min"),
        F.max("credit_score").alias("max"),
        F.expr(
            "percentile_approx(credit_score, array(0.01, 0.25, 0.5, 0.75, 0.99), 10000)"
        ).alias("percentiles"),
    )
    .orderBy("vintage").show()
)

+-------+-------+-----+-------------+---+----+--------------------+
|vintage|   rows|nulls|sentinel_9999|min| max|         percentiles|
+-------+-------+-----+-------------+---+----+--------------------+
|   2015|2741047|    0|           83|402|9999|[624, 717, 759, 7...|
|   2016|3233622|    0|          187|312|9999|[629, 716, 758, 7...|
|   2017|2623879|    0|          271|424|9999|[628, 712, 752, 7...|
|   2018|2371786|    0|          639|427|9999|[630, 713, 753, 7...|
|   2019|3063859|    0|         1264|309|9999|[635, 720, 757, 7...|
|   2020|7132087|    0|          959|398|9999|[645, 732, 769, 7...|
|   2021|7846853|    0|          873|300|9999|[636, 720, 760, 7...|
|   2022|3050733|    0|          589|300|9999|[628, 711, 751, 7...|
+-------+-------+-----+-------------+---+----+--------------------+



None

In [63]:
credit_score_invalid = (
    audit_df.filter(
        F.col("credit_score").isNotNull()
        & (F.col("credit_score") != 9999)
        & ((F.col("credit_score") < 300) | (F.col("credit_score") > 850))
    )
    .limit(1)
    .count()
)

print("Invalid usable credit scores:", credit_score_invalid)

Invalid usable credit scores: 0


In [76]:
df = spark.read.parquet("C:/Users/vorad/OneDrive/Desktop/Projects/mortgage-credit-risk/data/02_intermediate/freddie_mac/2015/performance")

In [77]:
df.columns

['loan_id',
 'period',
 'current_actual_upb',
 'current_loan_delinquency_status',
 'loan_age',
 'remaining_months_to_legal_maturity',
 'defect_settlement_date',
 'modification_flag',
 'zero_balance_code',
 'zero_balance_effective_date',
 'current_interest_rate',
 'current_non_interest_bearing_upb',
 'ddlpi',
 'mi_recoveries',
 'net_sales_proceeds',
 'non_mi_recoveries',
 'total_expenses',
 'legal_costs',
 'maintenance_preservation_costs',
 'taxes_and_insurance',
 'miscellaneous_expenses',
 'actual_loss',
 'cumulative_modification_costs',
 'interest_rate_step_indicator',
 'payment_deferral_flag',
 'estimated_ltv',
 'zero_balance_removal_upb',
 'delinquent_accrued_interest',
 'delinquency_due_to_disaster',
 'borrower_assistance_plan',
 'current_period_modification_costs',
 'current_interest_bearing_upb',
 'mi_cancellation_indicator',
 'servicer_name',
 'bankruptcy_cramdown_costs']